# IOAI — 2026 Contest Audio Event Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request, gzip, shutil
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-audio-event-detection'
for f in ['X_test.npy','train.csv','test_ids.csv']:
    if not os.path.exists(f'data/{f}'): urllib.request.urlretrieve(f'{BASE}/{f}', f'data/{f}')
if not os.path.exists('data/X_train.npy'):
    with open('xt.gz','wb') as out:
        for part in ['Xtr_part_00','Xtr_part_01']:
            urllib.request.urlretrieve(f'{BASE}/{part}', part)
            with open(part,'rb') as p: shutil.copyfileobj(p, out)
    with gzip.open('xt.gz','rb') as g, open('data/X_train.npy','wb') as o: shutil.copyfileobj(g, o)
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Audio Event Detection (APOAI 2026)

3초 16kHz 통화음에서 **음향 이벤트 8종**을 단일라벨 분류한다: cough·sneeze·laughter·cry(중요, 가중치 2.0),
dog_bark·siren·noise·none(가중치 1.0). 입력은 **사전계산된 log-mel 스펙트로그램** `[64,300]`. 사전학습 금지.
`data/X_train.npy`+`data/train.csv`(id,label)로 학습, `data/X_test.npy`(+`test_ids.csv`)를 예측해
`submission.json`={id: label}. 채점: 클래스별 F1 의 가중평균.

In [ ]:
import numpy as np, pandas as pd, json
Xtr = np.load("data/X_train.npy").astype("float32")   # [N,64,300]
train = pd.read_csv("data/train.csv")                 # id, label
Xte = np.load("data/X_test.npy").astype("float32")
test_ids = pd.read_csv("data/test_ids.csv")["id"].tolist()
CLASSES = ["cough","sneeze","laughter","cry","dog_bark","siren","noise","none"]
print(Xtr.shape, "| classes", train.label.nunique())

In [ ]:
# 베이스라인: 최빈 클래스 예측 — 가중 F1 낮음
maj = train.label.mode()[0]
json.dump({str(i): maj for i in test_ids}, open("submission.json","w"), indent=2)
print("saved submission.json", len(test_ids))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)